In [1]:
"""
Skenario 3: NeuMF Baseline + TPE
Arsitektur IDENTIK dengan Skenario 1 (sengaja lemah):
  - MLP hanya 2 layer: [emb_dim*2 → 64 → 32]
  - Tidak ada _init_weights / Xavier init
TPE mengoptimasi: emb_dim, lr, dropout, batch_size
  - TUNE_EPOCHS=10, N_TRIALS=25
  - Final training: EPOCHS=30 dengan best params
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from tqdm import tqdm
import math
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =====================
# CONFIG
# =====================
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 30
TUNE_EPOCHS  = 10
TOP_K        = 10
NUM_NEG      = 99
N_TRIALS     = 25
RANDOM_STATE = 42

print("=" * 50)
print("  Skenario 3: NeuMF Baseline + TPE")
print("=" * 50)
print("Device:", DEVICE)

# =====================
# LOAD DATA
# =====================
print("\nMemuat dataset...")
train_df = pd.read_csv("data/train_dataset.csv")
test_df  = pd.read_csv("data/test_dataset.csv")
full_df  = pd.read_csv("data/user_dataset_final.csv")

n_users = full_df['user_id_enc'].max() + 1
n_items = full_df['item_id_enc'].max() + 1
print(f"Users: {n_users}, Items: {n_items}")
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

user_positive_items = (
    train_df[train_df['label'] == 1]
    .groupby('user_id_enc')['item_id_enc']
    .apply(set)
    .to_dict()
)

# =====================
# DATASET (STATIC)
# =====================
dataset = TensorDataset(
    torch.tensor(train_df['user_id_enc'].values).long(),
    torch.tensor(train_df['item_id_enc'].values).long(),
    torch.tensor(train_df['label'].values).float()
)
print(f"Training samples: {len(dataset)}")

# =====================
# MODEL — IDENTIK DENGAN SKENARIO 1 (LEMAH)
# MLP 2 layer, tidak ada Xavier init
# =====================
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64, dropout=0.2):
        super().__init__()
        self.user_gmf = nn.Embedding(n_users, emb_dim)
        self.item_gmf = nn.Embedding(n_items, emb_dim)
        self.user_mlp = nn.Embedding(n_users, emb_dim)
        self.item_mlp = nn.Embedding(n_items, emb_dim)

        # MLP 2 layer (sama seperti Skenario 1)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 64),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.Dropout(dropout),
            nn.ReLU()
        )
        self.output = nn.Linear(emb_dim + 32, 1)
        # Tidak ada _init_weights

    def forward(self, user, item):
        gmf    = self.user_gmf(user) * self.item_gmf(item)
        mlp_in = torch.cat([self.user_mlp(user), self.item_mlp(item)], dim=-1)
        mlp    = self.mlp(mlp_in)
        x      = torch.cat([gmf, mlp], dim=-1)
        return self.output(x).squeeze()

# =====================
# EVALUASI — LOO + 99 NEGATIF
# =====================
@torch.no_grad()
def evaluate(model, seed=RANDOM_STATE):
    model.eval()
    hits, ndcgs = [], []
    rng = np.random.default_rng(seed)

    test_pos = test_df[test_df['label'] == 1].copy()

    for row in tqdm(test_pos.itertuples(index=False), total=len(test_pos),
                    desc="Evaluating", leave=False):
        user      = int(row.user_id_enc)
        true_item = int(row.item_id_enc)
        positives = user_positive_items.get(user, set())

        negatives = set()
        while len(negatives) < NUM_NEG:
            j = int(rng.integers(n_items))
            if j != true_item and j not in positives:
                negatives.add(j)

        items_eval = list(negatives) + [true_item]
        users_eval = [user] * len(items_eval)

        user_t = torch.tensor(users_eval).long().to(DEVICE)
        item_t = torch.tensor(items_eval).long().to(DEVICE)

        scores    = torch.sigmoid(model(user_t, item_t)).cpu().numpy()
        rank      = np.argsort(scores)[::-1]
        top_items = np.array(items_eval)[rank[:TOP_K]]

        if true_item in top_items:
            r = int(np.where(top_items == true_item)[0][0]) + 1
            hits.append(1)
            ndcgs.append(1.0 / math.log2(r + 1))
        else:
            hits.append(0)
            ndcgs.append(0.0)

    return np.mean(hits), np.mean(ndcgs)

# =====================
# OBJECTIVE FUNCTION — TPE
# Search space: emb_dim, lr, dropout, batch_size
# =====================
def objective(trial):
    emb_dim    = trial.suggest_categorical("emb_dim",    [32, 64, 128])
    lr         = trial.suggest_float("lr",               1e-4, 1e-2, log=True)
    dropout    = trial.suggest_float("dropout",          0.0, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [512, 1024, 2048])

    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_STATE)
    )
    torch.manual_seed(RANDOM_STATE)
    model     = NeuMF(n_users, n_items, emb_dim=emb_dim, dropout=dropout).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(TUNE_EPOCHS):
        model.train()
        for u, i, l in loader:
            u, i, l = u.to(DEVICE), i.to(DEVICE), l.to(DEVICE)
            pred = model(u, i)
            loss = criterion(pred, l)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    hr10, _ = evaluate(model, seed=RANDOM_STATE)
    return hr10

# =====================
# RUN OPTUNA TPE
# =====================
sampler = TPESampler(seed=RANDOM_STATE)
study   = optuna.create_study(
    direction  = "maximize",
    sampler    = sampler,
    study_name = "NCF_Baseline_TPE_Skenario3"
)

print(f"\nStarting Optuna TPE ({N_TRIALS} trials, TUNE_EPOCHS={TUNE_EPOCHS})...")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_trial.params
print("\nBest Trial:")
print(f"  Value (HR@10): {study.best_trial.value:.4f}")
for key, val in best_params.items():
    print(f"    {key}: {val}")

# Simpan semua trial ke CSV
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
study.trials_dataframe().to_csv(
    OUTPUT_DIR / "optuna_trials_skenario3.csv", index=False
)
print(f"Semua trial disimpan: {OUTPUT_DIR / 'optuna_trials_skenario3.csv'}")

# =====================
# FINAL TRAINING — EPOCHS=30
# =====================
torch.manual_seed(RANDOM_STATE)
final_loader = DataLoader(
    dataset, batch_size=best_params["batch_size"], shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_STATE)
)
final_model = NeuMF(
    n_users, n_items,
    emb_dim = best_params["emb_dim"],
    dropout = best_params["dropout"]
).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params["lr"])

print(f"\nTraining Final Model ({EPOCHS} epochs) dengan best params:")
for key, val in best_params.items():
    print(f"  {key}: {val}")
print()

for epoch in range(EPOCHS):
    final_model.train()
    total_loss = 0
    for u, i, l in final_loader:
        u, i, l = u.to(DEVICE), i.to(DEVICE), l.to(DEVICE)
        pred = final_model(u, i)
        loss = criterion(pred, l)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss:.4f}")

# =====================
# EVALUASI MULTI-SEED
# =====================
EVAL_SEEDS = [42, 123, 456]
print("\nEvaluasi Multi-Seed...")
hr_list, ndcg_list = [], []
for seed in EVAL_SEEDS:
    hr, ndcg = evaluate(final_model, seed=seed)
    hr_list.append(hr)
    ndcg_list.append(ndcg)
    print(f"  Seed {seed:3d} | HR@{TOP_K}={hr:.4f}, NDCG@{TOP_K}={ndcg:.4f}")

mean_hr   = float(np.mean(hr_list))
mean_ndcg = float(np.mean(ndcg_list))
std_hr    = float(np.std(hr_list))
std_ndcg  = float(np.std(ndcg_list))

print(f"\n{'='*50}")
print(f"  HASIL EVALUASI — Skenario 3: NeuMF Baseline + TPE")
print(f"{'='*50}")
print(f"  Mean HR@{TOP_K}    : {mean_hr:.4f} ± {std_hr:.4f}")
print(f"  Mean NDCG@{TOP_K}  : {mean_ndcg:.4f} ± {std_ndcg:.4f}")

# =====================
# SIMPAN HASIL
# =====================
result_df = pd.DataFrame([{
    "skenario"     : 3,
    "model"        : "NeuMF Baseline + TPE",
    "cluster"      : False,
    "tpe"          : True,
    "HR@10_mean"   : round(mean_hr, 4),
    "HR@10_std"    : round(std_hr, 4),
    "NDCG@10_mean" : round(mean_ndcg, 4),
    "NDCG@10_std"  : round(std_ndcg, 4),
    "seeds"        : str(EVAL_SEEDS),
    "best_params"  : str(best_params),
    "n_trials"     : N_TRIALS,
    "tune_epochs"  : TUNE_EPOCHS,
    "epochs"       : EPOCHS,
    "notes"        : "MLP 2 layer, no Xavier init, TPE optimized"
}])

result_path = OUTPUT_DIR / "hasil_skenario3_baseline_tpe.csv"
result_df.to_csv(result_path, index=False)
print(f"\nHasil disimpan: {result_path}")

Path("models").mkdir(exist_ok=True)
torch.save(final_model.state_dict(), "models/skenario3_baseline_tpe.pt")
print("Model disimpan: models/skenario3_baseline_tpe.pt")

d:\TA_NCF+K-means\Code\NCF_Cluster\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Skenario 3: NeuMF Baseline + TPE
Device: cuda

Memuat dataset...
Users: 9529, Items: 4144
Train size: 2758490, Test size: 952900
Training samples: 2758490

Starting Optuna TPE (25 trials, TUNE_EPOCHS=10)...


Best trial: 23. Best value: 0.836289: 100%|██████████| 25/25 [4:35:31<00:00, 661.27s/it]  



Best Trial:
  Value (HR@10): 0.8363
    emb_dim: 32
    lr: 0.0090060204999397
    dropout: 0.24187626008357324
    batch_size: 512
Semua trial disimpan: outputs\optuna_trials_skenario3.csv

Training Final Model (30 epochs) dengan best params:
  emb_dim: 32
  lr: 0.0090060204999397
  dropout: 0.24187626008357324
  batch_size: 512

Epoch 01/30 | Loss: 2114.9675
Epoch 02/30 | Loss: 1848.3626
Epoch 03/30 | Loss: 1552.5434
Epoch 04/30 | Loss: 1327.0440
Epoch 05/30 | Loss: 1164.5785
Epoch 06/30 | Loss: 1055.5628
Epoch 07/30 | Loss: 978.8652
Epoch 08/30 | Loss: 921.6480
Epoch 09/30 | Loss: 876.5552
Epoch 10/30 | Loss: 842.6828
Epoch 11/30 | Loss: 815.1261
Epoch 12/30 | Loss: 791.0458
Epoch 13/30 | Loss: 771.5709
Epoch 14/30 | Loss: 754.1281
Epoch 15/30 | Loss: 738.0179
Epoch 16/30 | Loss: 725.2869
Epoch 17/30 | Loss: 712.4795
Epoch 18/30 | Loss: 702.9046
Epoch 19/30 | Loss: 691.7941
Epoch 20/30 | Loss: 681.2694
Epoch 21/30 | Loss: 674.1709
Epoch 22/30 | Loss: 666.4064
Epoch 23/30 | Loss: 65

  Seed  42 | HR@10=0.8172, NDCG@10=0.5606


  Seed 123 | HR@10=0.8176, NDCG@10=0.5616


  Seed 456 | HR@10=0.8117, NDCG@10=0.5566

  HASIL EVALUASI — Skenario 3: NeuMF Baseline + TPE
  Mean HR@10    : 0.8155 ± 0.0027
  Mean NDCG@10  : 0.5596 ± 0.0022

Hasil disimpan: outputs\hasil_skenario3_baseline_tpe.csv
Model disimpan: models/skenario3_baseline_tpe.pt
